# FairMALLM-GAN v3 — SMOKE TEST (~30 min)

Settings: **Accelerator = GPU T4 x2**, **Internet = on**, inputs **oulad-processed** + **fairmallm-v3-code**.

Run in the background: *Save Version → Save & Run All (Commit)*. Results appear in the version's **Output** tab (`run_*/` = synthetic data, `eval_*/` = evaluation).

To run another configuration change `MODE` below: `full` (proposed model), `no_fair` (ablation without D2), `mallm_flat` (MALLM-GAN baseline).

In [ ]:
!pip install -q transformers accelerate


In [ ]:
import glob, shutil
for f in ["common.py", "fair_mallm_gan_v3.py", "evaluate_v3.py", "baseline_ctgan.py", "prepare_cohort.py"]:
    src = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
    assert src, f"{f} not found - attach the dataset 'fairmallm-v3-code' (Add Input)"
    shutil.copy(src[0], f"/kaggle/working/{f}")
print("code copied")


In [ ]:
import glob, os, sys, json, shutil, torch
sys.path.insert(0, "/kaggle/working"); os.chdir("/kaggle/working")
hits = glob.glob("/kaggle/input/**/oulad_processed/X_seq.npy", recursive=True) or \
       glob.glob("/kaggle/input/**/X_seq.npy", recursive=True)
assert hits, "Attach the dataset 'oulad-processed' (Add Input) first."
DATA_DIR = os.path.dirname(hits[0])
print("DATA_DIR =", DATA_DIR)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
MODE = "full"            # full | no_fair | mallm_flat
from fair_mallm_gan_v3 import Config, run
cfg = Config(DATA_DIR=DATA_DIR, OUT_DIR=f"/kaggle/working/run_{MODE}", mode=MODE,
             cohort="active", n_synthetic=64, max_rounds=2,
             prompts_per_batch=8, students_per_prompt=8, time_budget_h=0.6)
synth, state = run(cfg)


In [ ]:
# quick evaluation of THIS run (full comparison: evaluate_all notebook)
from evaluate_v3 import run as run_eval
res = run_eval(DATA_DIR, f"/kaggle/working/eval_{MODE}", {MODE: cfg.OUT_DIR},
               cohort="active", seeds=(0,))
print(open(f"/kaggle/working/eval_{MODE}/report.md").read())


In [ ]:
for root, _, fs in os.walk("/kaggle/working"):
    print(root, sorted(fs)[:12])
